[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sampriti-sr/MLP_Course/blob/kshitij-variant/docs/lifecycle/14-reproducibility.ipynb)

# Reproducibility and Project Hygiene

<div class="admonition info">
<p class="admonition-title">What you will be able to do after this page</p>
<ul>
<li>Prove that a whole project reproduces, including from a fresh process.</li>
<li>Record a run so that it can be re-created from its manifest alone.</li>
<li>Generate a model card and a decision log from artifacts, not memory.</li>
<li>Map every trap you met to a test that guards it.</li>
</ul>
</div>

**Prerequisites:** <a href="../13-monitoring/">Monitoring and Drift</a>. **Dataset:** Online Retail II. **Time:** about 45 minutes.

In [1]:
try:
    import mlp_data
except ImportError:  # e.g. a fresh Colab runtime
    %pip install -q "mlp-course @ git+https://github.com/sampriti-sr/MLP_Course@kshitij-variant"
    import mlp_data

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mlp_data import load_retail, retail, style

style.apply()
pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 14)
pd.set_option("display.float_format", "{:,.2f}".format)

## The situation

Imagine a colleague opens this project in six months and is asked: *"Where did the 340,000 come from, and can you reproduce it?"* Or the regulator asks. Or **you** are that colleague, and you have forgotten everything. A result that cannot be re-created is a rumour.

Reproducibility is not one big feature. It is a habit built from small, boring, checkable things: pinned inputs, one place for settings, controlled randomness, recorded environments, tests that guard the traps we hit, and writing down *why*. This last page of the lifecycle turns each of those from a good intention into something we can run and check.

## 1. Can the whole project be re-run, and get the same answer?

`mlp_data.pipeline` is the entire project as one function: raw data in, a result out. Everything that can change a number lives in a small `Config`; everything that identifies what was run goes in a **manifest**. We run it twice in this notebook, and then once more in a **brand-new Python process**, to rule out hidden state such as variables left over from earlier cells.

In [2]:
import json, os, subprocess, sys, tempfile
from importlib import metadata
from IPython.display import Markdown, display
from mlp_data import models, serving
from mlp_data.pipeline import Config, run_pipeline
from mlp_data.tracking import ExperimentLog, fingerprint, environment

first = run_pipeline(Config())
second = run_pipeline(Config())

fresh = subprocess.run([sys.executable, "-m", "mlp_data.pipeline"], capture_output=True, text=True, cwd=tempfile.gettempdir(), env=os.environ, timeout=900)
assert fresh.returncode == 0, fresh.stderr[-500:]
third = json.loads(fresh.stdout)

pd.DataFrame({"this notebook, run 1": first["predictions"], "this notebook, run 2": second["predictions"], "fresh Python process": third["predictions"]}, index=["fingerprint of the predictions"]).T

,fingerprint of the predictions
"this notebook, run 1",fe7f3fda68e5
"this notebook, run 2",fe7f3fda68e5
fresh Python process,fe7f3fda68e5


In [3]:
assert first["predictions"] == second["predictions"] == third["predictions"]
assert first["metrics"] == second["metrics"] == third["metrics"]
print("identical predictions and metrics across all three runs")
fmt = lambda name, v: f"{v:,.0f}" if name.startswith("revenue@") and "oracle" not in name else (f"{v:.1%}" if "%" in name or name.startswith("precision") else f"{v:.3f}")
pd.Series({name: fmt(name, v) for name, v in first["metrics"].items()}, name="test result").to_frame()

identical predictions and metrics across all three runs


,test result
AUC,0.767
Brier score,0.227
log loss,0.656
precision@200,28.0%
revenue@200,"340,926"
revenue@200 % of oracle,49.6%


Three runs, one fingerprint, and the headline numbers match the test verdict on the packaging page (AUC about 0.77, revenue protected about 340,000, roughly half the best possible). Anyone with the same code, the same data and the same library versions gets the same answer, and the fingerprint is how they *know*.

## 2. What exactly was it run on? The manifest

A fingerprint is only useful if you can compare it with something. The manifest records the configuration, a fingerprint of each dataset, the results, and the environment, which is everything needed to say "that number came from *this*".

In [4]:
show = {"config": first["config"], "data fingerprints": first["data"], "predictions fingerprint": first["predictions"], "environment": first["env"]}
print(json.dumps(show, indent=2))

{
  "config": {
    "C": 1.0,
    "overdue_cap": 50,
    "k": 200,
    "seed": 0
  },
  "data fingerprints": {
    "cleaned_sales": "71cb594a10e4",
    "train": "de610f93a4c9",
    "valid": "e0eb1c938a1c",
    "test": "a9be028b046b"
  },
  "predictions fingerprint": "fe7f3fda68e5",
  "environment": {
    "python": "3.14.2",
    "numpy": "2.5.4",
    "pandas": "3.0.6",
    "scikit-learn": "1.9.1"
  }
}


### The environment

"Same library versions" needs checking, not assuming. These are the versions installed *right now* for the packages this project depends on. In a real project this list is saved next to the results (a lock file, or `pip freeze > requirements.lock`) so the environment can be rebuilt.

In [5]:
packages = ["mlp-course", "pandas", "numpy", "scikit-learn", "scipy", "matplotlib", "pyarrow", "joblib", "openpyxl", "xlrd"]
versions = {}
for name in packages:
    try:
        versions[name] = metadata.version(name)
    except metadata.PackageNotFoundError:
        versions[name] = "(not installed)"
pd.Series(versions, name="installed version").to_frame()

,installed version
mlp-course,0.1.0
pandas,3.0.6
numpy,2.5.4
scikit-learn,1.9.1
scipy,1.18.1
matplotlib,3.11.2
pyarrow,26.0.0
joblib,1.6.0
openpyxl,3.1.5
xlrd,2.0.2


## 3. Controlled randomness: how much does a seed matter?

Randomness enters in a few places: tie-breaking in the ranking, the bootstrap intervals, the random search, and (for forests) the trees themselves. Each must have a **recorded seed**. But it is also worth knowing how large the effect of a seed *is*, so that you know whether a difference you are chasing is bigger than the noise from randomness alone.

In [6]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

cleaned = retail.clean_retail(load_retail())
sp = retail.build_splits(cleaned.sales, cleaned.returns, features="full")
F = models.FEATURES
tree_input = lambda d: d[F].fillna(-1)

aucs = []
for seed in range(6):
    forest = RandomForestClassifier(n_estimators=200, min_samples_leaf=20, n_jobs=-1, random_state=seed).fit(tree_input(sp.train), sp.train["y"])
    aucs.append(roc_auc_score(sp.valid["y"], forest.predict_proba(tree_input(sp.valid))[:, 1]))
logistic_aucs = {s: roc_auc_score(sp.valid["y"], models.make_logistic().fit(sp.train[F], sp.train["y"]).predict_proba(sp.valid[F])[:, 1]) for s in range(2)}

print(f"random forest, six seeds: validation AUC from {min(aucs):.4f} to {max(aucs):.4f} (standard deviation {np.std(aucs):.4f})")
print(f"logistic regression (no randomness in fitting): {logistic_aucs[0]:.4f} and {logistic_aucs[1]:.4f}, identical: {logistic_aucs[0] == logistic_aucs[1]}")

random forest, six seeds: validation AUC from 0.7955 to 0.7979 (standard deviation 0.0008)
logistic regression (no randomness in fitting): 0.7959 and 0.7959, identical: True


**Reading it.** The forest's seed noise is **small but not zero**: across six seeds the AUC moves by about two thousandths from best to worst (a standard deviation under a thousandth). The logistic regression has no randomness in its fitting, so it gives the same answer every time.

Two uses. First, it sets a **noise floor**: a gap of a couple of thousandths between two *randomised* models is inside the seed noise and means nothing; the gaps between model *families* on the modelling page were several times larger. Second, it is a reason to prefer a deterministic model where it is good enough, and to **record the seed** wherever randomness enters.

## 4. A manifest is a recipe

The proof of a record is being able to follow it. Save a run to a log file, forget the notebook, read the file back, and re-create the result from nothing but what it says.

In [7]:
path = tempfile.NamedTemporaryFile(suffix=".jsonl", delete=False).name
log = ExperimentLog(path)
log.log("final logistic, test verdict", first["config"], first["metrics"], seed=first["config"]["seed"], data=first["data"]["cleaned_sales"], notes="the shipped model")

recipe = ExperimentLog(path).runs[0]                     # a new session reads the file back
rebuilt = run_pipeline(Config(**recipe["params"]))      # ... and re-creates the run from its recorded configuration alone
assert rebuilt["predictions"] == first["predictions"]
print("re-created from the log alone; predictions fingerprint", rebuilt["predictions"], "matches the original")

re-created from the log alone; predictions fingerprint fe7f3fda68e5 matches the original


## 5. Tests: a guard for every trap we met

The course is, among other things, a record of traps. Each deserves a test that fails if it ever comes back. Here is the map of what the project's own test suite guards, and where in the course the trap appeared:

| The trap | The test that guards it | Met on |
|---|---|---|
| The two Excel sheets overlap and double-count a week | a synthetic overlap is removed exactly once | Cleaning |
| A cancelled order counts as a purchase | a fully reversed pair is netted out; partial and late cancellations are kept | Cleaning |
| The ledger stops reconciling to the raw file | rows removed plus kept equals rows in | Cleaning |
| A feature uses the future | **features unchanged when future data changes**; the label window is half-open | Feature engineering |
| Train and validation windows overlap | the split checker rejects overlapping label windows | Splitting |
| Ties in a ranking decided by file order | tie-breaking independent of row order | Baselines |
| A metric that is optimistic by construction | exact random baseline; paired bootstrap detects a real difference | Baselines |
| A model that cannot be saved safely | save/load round trip; **a tampered file is refused** | Serving |
| Silent bad input | missing columns, bad dates and stale data fail with a clear message | Serving |
| Serving differs from the batch evaluation | serving reproduces the batch scores; future data cannot change a score | Serving |
| The project stops being reproducible | same config and data give identical fingerprints | this page |

If the course repository is present (it is when you run from a clone), the suite can be run right here.

In [8]:
from pathlib import Path
tests_dir = Path(retail.__file__).resolve().parent.parent / "tests"
if tests_dir.exists():
    out = subprocess.run([sys.executable, "-m", "pytest", "-q", "--no-header", "--color=no", "-p", "no:cacheprovider", str(tests_dir)], capture_output=True, text=True, timeout=600)
    print(out.stdout.strip().splitlines()[-1])
else:
    print("(the tests live in the course repository's tests/ folder; clone it to run them)")

45 passed in 3.97s


## 6. Say what it is: a model card generated from the facts

A model card is a short document that travels with a model: what it is for, what it was built from, how well it works, and **where it should not be used**. The commonest failure is writing one by hand and letting it drift from the truth. So we generate it from the artifacts we already have: the manifest, the test results and the findings recorded on earlier pages.

In [9]:
labelled = pd.concat([sp.train, sp.valid], ignore_index=True)
final = models.make_logistic().fit(labelled[F], labelled["y"])
p_test = final.predict_proba(sp.test[F])[:, 1]
card_dir = Path(tempfile.mkdtemp(prefix="mlp_card_"))
saved = serving.save_model(final, card_dir, "repurchase_logistic_v1", metrics=first["metrics"], training={"rows": int(len(labelled))})
m = first["metrics"]

card = f"""
# Model card: repurchase-risk ranking, version 1

**Purpose.** Rank the customers of a UK gift-ware wholesaler by *value at risk* (the chance they do not buy again in the next 90 days, times their past-year spend), so that account managers can phone the top {first['config']['k']} each month.

**Not for.** Quoting the probability of a customer returning as a forecast, especially in a peak season (see Limitations). Not for any decision about an individual other than prioritising outreach.

**Data.** Online Retail II, UCI Machine Learning Repository (CC BY 4.0): invoices from December 2009 to December 2011. Identified customers only; guest checkouts (about 13% of product revenue) are excluded because repeat behaviour cannot be tracked.

**Training.** Logistic regression on {len(models.FEATURES)} behavioural features, fitted on {len(labelled):,} customer-snapshots (snapshots on 1 Dec 2010 to 1 Mar 2011, and 1 Jun 2011), with a 90-day embargo before the test window.

**Test result (snapshot 1 Sep 2011, {len(sp.test):,} customers; scored once).**
- Ranking quality: AUC **{m['AUC']:.3f}**.
- Revenue protected within {first['config']['k']} calls: **{m[f"revenue@{first['config']['k']}"]:,.0f}** ({m[f"revenue@{first['config']['k']} % of oracle"]:.0%} of the best possible). A free no-model rule achieved about the same: the model is **not demonstrably better** than the rule on money.
- Calibration: the model predicted a **{p_test.mean():.0%}** return rate; **{sp.test['y'].mean():.0%}** actually returned.

**Limitations.**
1. Probabilities were far too low in the autumn peak; it had seen one winter and one early summer. Use the *order*, not the *odds*.
2. New customers (under 90 days) were under-predicted by about 28 points on test.
3. It cannot anticipate regular customers who suddenly stop; a different algorithm did not fix this (the information is missing).
4. One season of history: seasonal effects are not modelled.

**Fairness considerations.** The model ignores country (negligible weight), but value-ranking sends all calls to the top spend quarter, and overseas customers receive about twice the call rate of UK customers at similar spend because of behaviour that differs by country. Customers outside the top spend quarter receive no calls. Whether that is acceptable is a policy decision; a low-touch channel for the remainder is recommended.

**Monitoring.** Early 30-day return rate (alarm at z above 3), matured calibration gap, model-versus-rule benchmark, seasonal calendar. Retrain quarterly and before known peaks, through the same gates.

**Provenance.** Model file hash `{saved['sha256'][:16]}...`; data fingerprints {first['data']['train']} (train), {first['data']['valid']} (validation), {first['data']['test']} (test); predictions fingerprint {first['predictions']}; Python {first['env']['python']}, scikit-learn {first['env']['scikit-learn']}, pandas {first['env']['pandas']}.
"""
display(Markdown(card))


# Model card: repurchase-risk ranking, version 1

**Purpose.** Rank the customers of a UK gift-ware wholesaler by *value at risk* (the chance they do not buy again in the next 90 days, times their past-year spend), so that account managers can phone the top 200 each month.

**Not for.** Quoting the probability of a customer returning as a forecast, especially in a peak season (see Limitations). Not for any decision about an individual other than prioritising outreach.

**Data.** Online Retail II, UCI Machine Learning Repository (CC BY 4.0): invoices from December 2009 to December 2011. Identified customers only; guest checkouts (about 13% of product revenue) are excluded because repeat behaviour cannot be tracked.

**Training.** Logistic regression on 18 behavioural features, fitted on 21,234 customer-snapshots (snapshots on 1 Dec 2010 to 1 Mar 2011, and 1 Jun 2011), with a 90-day embargo before the test window.

**Test result (snapshot 1 Sep 2011, 4,323 customers; scored once).**
- Ranking quality: AUC **0.767**.
- Revenue protected within 200 calls: **340,926** (50% of the best possible). A free no-model rule achieved about the same: the model is **not demonstrably better** than the rule on money.
- Calibration: the model predicted a **32%** return rate; **50%** actually returned.

**Limitations.**
1. Probabilities were far too low in the autumn peak; it had seen one winter and one early summer. Use the *order*, not the *odds*.
2. New customers (under 90 days) were under-predicted by about 28 points on test.
3. It cannot anticipate regular customers who suddenly stop; a different algorithm did not fix this (the information is missing).
4. One season of history: seasonal effects are not modelled.

**Fairness considerations.** The model ignores country (negligible weight), but value-ranking sends all calls to the top spend quarter, and overseas customers receive about twice the call rate of UK customers at similar spend because of behaviour that differs by country. Customers outside the top spend quarter receive no calls. Whether that is acceptable is a policy decision; a low-touch channel for the remainder is recommended.

**Monitoring.** Early 30-day return rate (alarm at z above 3), matured calibration gap, model-versus-rule benchmark, seasonal calendar. Retrain quarterly and before known peaks, through the same gates.

**Provenance.** Model file hash `139b73b8b494b66f...`; data fingerprints de610f93a4c9 (train), e0eb1c938a1c (validation), a9be028b046b (test); predictions fingerprint fe7f3fda68e5; Python 3.14.2, scikit-learn 1.9.1, pandas 3.0.6.


Every number in that card was read from a variable, so if the data or the model changes, the card changes with it. Compare it with a hand-written paragraph that says "the model performs well": the card says what it is for, what it is not for, how it was tested, and what would make you stop trusting it.

## 7. The decision log

Pages of reasoning are easy to lose. A **decision log** records each choice with the evidence behind it, in one place a newcomer can scan. The ones this project made:

| # | Decision | Evidence | Page |
|---|---|---|---|
| 1 | Predict "buys again within **90 days**" for customers active in the past year | 81% of reorder gaps are 90 days or shorter | Problem framing |
| 2 | Snapshots start **1 Dec 2010** | Earlier snapshots are left-censored; the data starts Dec 2009 | Problem framing, EDA |
| 3 | **Chronological split with a 90-day embargo**: train Dec to Mar, validate Jun, test Sep | A random split flattered a flexible model by about +0.07 AUC | Splitting |
| 4 | Clean in six accounted steps; exclude guest checkouts; **net out orders cancelled within a day** | Cancelled orders left labels untouched but distorted spend | Cleaning |
| 5 | Judge by **revenue protected within a call budget**, alongside AUC | Accuracy-optimised rankings protected less than random calling | Baselines |
| 6 | Rank by **value at risk**, not probability | Same model: ~2% of the best possible revenue vs ~58% | Baselines |
| 7 | Keep all features | Small real AUC gain; no demonstrable money gain | Features |
| 8 | **Logistic regression** is the final model | Nothing beat it on both ranking and money | Modelling, Tuning |
| 9 | Ship the model **with the simple rule as a benchmark** | On the test window they were tied on money | Packaging |
| 10 | Monitor with an early partial-label rate, a benchmark guard and the seasonal calendar | PSI did not see the autumn shift; the early rate did, a month before any full outcome | Monitoring |

## 8. Handing it over

A project is finished when someone else can run it, check it and change it. A last checklist:

- [ ] The repository contains **no data and no secrets**; data is fetched and checksummed
- [ ] One command re-creates the headline result (`python -m mlp_data.pipeline`), and its fingerprint is recorded
- [ ] Every setting lives in one `Config`; every random choice has a recorded seed
- [ ] The environment is captured (versions or a lock file)
- [ ] Tests run in CI on every change, including the leakage and serving-consistency tests
- [ ] The model file is stored with its card and hash, and the previous version is kept for rollback
- [ ] The model card states intended use, limitations, fairness findings and the monitoring plan
- [ ] The decision log is current, and negative results are kept
- [ ] There is a named owner, a retraining schedule and an alert policy

## What the lifecycle taught

Follow one project from a vague request to a documented, monitored model and a theme appears again and again. **The expensive mistakes were not modelling mistakes.** They were a wrong question, a leaky split, an unexamined cleaning choice, a metric that did not match the decision, a headline number that did not survive a fair test, and a world that changed after launch. The modelling itself was, in the end, a small part: a logistic regression was good enough, and a free rule nearly matched it.

## Check yourself

<details class="question">
<summary>1. A fresh Python process gave the same fingerprint as the notebook. What would it have meant if it had not?</summary>

That the result depended on something other than the code, config and data: a variable left over from an earlier cell, a package version, a file that differs, or an unseeded random number. Running in a clean process is the cheapest way to flush out hidden state, which is why it belongs in the routine.

</details>

<details class="question">
<summary>2. The random forest's AUC varies a little between seeds. Why record the seed anyway?</summary>

Because "a little" is still the difference between reproducing a number and not, and because differences smaller than the seed noise are not evidence of anything. Recording the seed makes a run repeatable; measuring the seed spread tells you how large a difference must be before it means something.

</details>

<details class="question">
<summary>3. Why generate the model card from variables instead of writing it by hand?</summary>

A hand-written card is true on the day it is written and drifts afterwards, quietly, as the model and data change. Numbers read from the artifacts cannot disagree with them. The prose (purpose, limitations) still needs a human, but anything checkable should be produced, not typed.

</details>

## Exercise

<details class="question">
<summary>Change one setting (the regularisation strength) and re-run the pipeline. What should happen to the predictions fingerprint, and what should <em>not</em> change?</summary>

```python
changed = run_pipeline(Config(C=0.001))
print("predictions fingerprint changed:", changed["predictions"] != first["predictions"])
print("data fingerprints unchanged:   ", changed["data"] == first["data"])
```

The predictions fingerprint **changes** (a different model makes different predictions) while the data fingerprints are **identical** (you altered the model, not the data). The pair tells you instantly *what* kind of change a result reflects, which is exactly what a manifest is for.

</details>

## Recap

- Reproducibility is a handful of checkable habits: pinned inputs, one `Config`, recorded seeds, a captured environment and a **fingerprint** that proves two runs agree.
- **Run in a fresh process** to rule out hidden state. A manifest should be enough to rebuild a run.
- Measure the noise from randomness so you know how big a difference has to be to matter.
- Keep a **test for every trap you met**, especially leakage and serving consistency.
- **Generate** the model card and decision log from facts, and record limitations and fairness findings alongside the numbers.
- The lesson of the whole lifecycle: the hard, expensive mistakes are upstream and downstream of the model. Spend your care there.

That completes the lifecycle. The next parts take what we built on the retail data and widen it: classical models in more depth, **time-series forecasting** on taxi demand, and **recommender systems** on ratings and purchases.